In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import InMemoryVectorStore
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain.agents import create_agent
from langchain.tools import tool

In [ ]:
loader = PyPDFLoader("../data/medical_report.pdf")
docs = loader.load()


In [ ]:
len(docs)

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splitted_docs = splitter.split_documents(docs)


In [ ]:
len(splitted_docs)

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = InMemoryVectorStore.from_documents(splitted_docs, embeddings)

In [ ]:
@tool
def retriever_tool(query: str) :
    """This tool can help you to retrieve the relevant data of the PDF Documents, and these pdf
        documents have details about medical reports."""

    docs = vector_store.similarity_search(query)
    context = ""
    for doc in docs:
        context += doc.page_content + "\n"
    return context

In [ ]:
llm = ChatGroq(
    model="openai/gpt-oss-20b"
)

In [ ]:
System_Prompt = """You are a helpful assistant that answers questions using retrieved context.
	ALWAYS use the `retriever_tool` tool for questions requiring external knowledge. """

In [ ]:
agent = create_agent(
    model=llm,
    tools=[retriever_tool],
    system_prompt=System_Prompt)


In [ ]:
query= " what is patient name and doctor name in the report?"
response = agent.invoke({"messages": [{"role": "user", "content": query}]})
result = response["messages"][-1].content

In [ ]:
print(result)